## About this notebook:

This notebook uses Affinder napari plugin to find transformation manually.

Input:
- the experiment's `experiment.json`: folders, wells and the anchor round every other round is aligned onto
- the well and round to correct (set below, as named in the folders)

Output:
- the well's transforms (`tmat_<well>.npy`, from notebook 02) with that round's transform replaced

Optionally you can visualize alignment (visualization uses Napari).

## Choose the experiment

Set `CONFIG` to the experiment's `experiment.json`. All folders and the well list come from it; create or edit it with `setup_experiment.ipynb`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

# the well and round to correct by hand
myWell = exp.well_list[0]  # change to the well to correct, e.g. 'D03'
myRound = 1  # as named in the folders: 1 for Round_R1

## Prepare for processing

In [ ]:
import os

import napari
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

from fouri import read_labels
from fouri.align import load_transforms, save_transforms


In [ ]:
# open df
df_path = os.path.join(exp.output_df, f'df_{myWell}.pkl')
df = pd.read_pickle(df_path)

# Print the unique values in the nameRound column to verify myRound is present
print("Unique values in 'nameRound':", df['nameRound'].unique())

# Convert myRound to the correct string format
myRound_str = f'R{myRound}'

# Verify that myRound is now in the correct format
print("Formatted 'myRound':", myRound_str)

# Check which alignRound for the selected round
filtered_align_round = df.loc[df.nameRound == myRound_str, 'alignRound']
print("Filtered 'alignRound' for 'myRound_str':", filtered_align_round)

align_round_list = filtered_align_round.tolist()

if align_round_list:
    alignRound = align_round_list[0]
    print(f"Selected alignRound: {alignRound}")
else:
    raise ValueError(f"No alignRound found for myRound = {myRound_str}. Please check the input data.")

# The round it is aligned onto is the experiment's anchor round, the one 02 and 06 use
# (the first round unless experiment.json sets "anchor_round"). It has nothing to correct.
anchor_round = exp.anchor_round
if alignRound == anchor_round:
    raise ValueError(
        f"{myRound_str} is the anchor round, which the other rounds are aligned onto. "
        "Set myRound to the round to correct."
    )
print(f"Anchor round: {anchor_round}")

# Open labels and the control
labels_im = read_labels(exp.segmented, myWell, indices=[anchor_round, alignRound])

# Open the list of transformations for a given well
tmat = load_transforms(exp, myWell)

# Select the appropriate transformation
t_org = tmat[int(alignRound)]


## Use Affinder in Napari

Remember to set parameters of the plugin correctly:
- reference image
- image to transform
- 'similarity' as transformation

Information about Affinder can be found:
- https://github.com/jni/affinder
- https://www.napari-hub.org/plugins/affinder?imageModality=Fluorescence+microscopy

In [ ]:
viewer = napari.Viewer()
l0 = viewer.add_image(labels_im[0], colormap='bop blue', blending='additive',name='ref')
l1 = viewer.add_image(labels_im[1], colormap='bop purple', blending='additive',name='moving')

qtwidget, widget = viewer.window.add_plugin_dock_widget(
        'affinder', 'Start affinder'
        )
widget.reference.bind(l0)
widget.moving.bind(l1)
widget()

In [ ]:
mat = np.asarray(viewer.layers["moving"].affine)

## Visualize transformation from the collected matrix (optional)

In [ ]:
# optionally - test matrix
tfd_ndi = ndi.affine_transform(labels_im[1], np.linalg.inv(mat))
viewer.add_image(tfd_ndi, colormap='bop orange', blending='additive')

## Save transformation matrix

In [ ]:
def matrix_rc2xy(affine_matrix):
    swapped_cols = affine_matrix[:, [1, 0, 2]]
    swapped_rows = swapped_cols[[1, 0, 2], :]
    return swapped_rows


tmat[int(alignRound),:,:] = np.linalg.inv(matrix_rc2xy(mat))
saved_path = save_transforms(exp, myWell, tmat)
print(f'Saved the corrected transforms to {saved_path}')